In [ ]:
from datasets import load_dataset

try:
    dataset = load_dataset("UKPLab/liar")
    print("\n✅ Successfully loaded 'liar' from UKPLab repository.")
    print(dataset)
except Exception as e:
    print(f"\n❌ Failed to load 'UKPLab/liar'. Error: {e}")


!pip install datasets transformers accelerate -q
!pip install --upgrade accelerate -q

import torch
from datasets import load_dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments
)

print("Dataset sample:", dataset["train"][0])

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def preprocess(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

dataset = dataset.map(preprocess, batched=True)

dataset = dataset.remove_columns(
    ["text", "label_text", "context"]
)

dataset.set_format("torch")

num_labels = 6

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels
)

collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./distilbert_misinfo",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    learning_rate=3e-5,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
    push_to_hub=False,
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=tokenizer,
    data_collator=collator,
)

trainer.train()

metrics = trainer.evaluate()
print("Evaluation metrics:", metrics)

trainer.save_model("./distilbert_misinfo")
tokenizer.save_pretrained("./distilbert_misinfo")

print("Training complete and model saved!")

In [ ]:
import shutil
shutil.make_archive("distilbert_misinfo_output", 'zip', "./distilbert_misinfo")

from google.colab import files
files.download("distilbert_misinfo_output.zip")

In [ ]:

!pip install datasets transformers accelerate -q
!pip install --upgrade accelerate -q
!pip install evaluate scikit-learn -q

import torch
from datasets import load_dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments
)
import numpy as np
import evaluate

accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    """
    Computes accuracy for a given set of predictions.
    """

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    return accuracy_metric.compute(predictions=predictions, references=labels)

num_labels = 6
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels
)

collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./distilbert_misinfo_run3",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
    push_to_hub=False,
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()

metrics = trainer.evaluate()
print("Evaluation metrics:", metrics)

trainer.save_model("./distilbert_misinfo_run3")
tokenizer.save_pretrained("./distilbert_misinfo_run3")

print("Training complete and model saved to ./distilbert_misinfo_run3!")